In [3]:
import torch
import torch.nn as nn

# 재현 가능한 결과를 위한 난수 시드
torch.manual_seed(2)

# ----------------------
# 1. 입력 데이터와 실제 정답
# ----------------------

X = torch.tensor([
    [0.5, 0.3, 0.8],
    [0.2, 0.0, 0.3],
    [0.8, 0.4, 1.0],
    [0.3, 0.1, 0.5]
], dtype=torch.float32)

y = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [0.0]
])

# ----------------------
# 2. DNN 모델 정의
# ----------------------

model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),

    nn.Linear(4, 2),
    nn.ReLU(),

    nn.Linear(2, 1)
)

# 이진 분류 손실 함수
loss_fn = nn.BCEWithLogitsLoss()

# 학습 모드
model.train()

# ----------------------
# 3. 순전파
# ----------------------

# 기존 Gradient 초기화
model.zero_grad()

# 현재 가중치 저장
weight_before = model[0].weight.detach().clone()

# Logit 계산
logits = model(X)

# 손실 계산
loss = loss_fn(logits, y)

print("입력 Shape:", tuple(X.shape))
print("Logit Shape:", tuple(logits.shape))
print("손실:", round(loss.item(), 4))

# ----------------------
# 4. 역전파
# ----------------------

loss.backward()

# ----------------------
# 5. Gradient 확인
# ----------------------

for name, parameter in model.named_parameters():
    print(
        f"{name} | "
        f"파라미터 Shape: {tuple(parameter.shape)} | "
        f"Gradient Shape: {tuple(parameter.grad.shape)} | "
        f"Gradient 크기: {parameter.grad.norm().item():.5f}"
    )

# ----------------------
# 6. 파라미터 변경 여부 확인
# ----------------------

weight_after = model[0].weight.detach()

is_changed = not torch.equal(
    weight_before,
    weight_after
)

print("역전파 후 가중치 변경 여부:", is_changed)

# 실행 결과 예시
# 입력 Shape: (4, 3)
# Logit Shape: (4, 1)
# 손실: 0.7115
# 0.weight | 파라미터 Shape: (4, 3) | Gradient Shape: (4, 3) | Gradient 크기: 0.06920
# 0.bias | 파라미터 Shape: (4,) | Gradient Shape: (4,) | Gradient 크기: 0.05926
# 2.weight | 파라미터 Shape: (2, 4) | Gradient Shape: (2, 4) | Gradient 크기: 0.04049
# 2.bias | 파라미터 Shape: (2,) | Gradient Shape: (2,) | Gradient 크기: 0.04378
# 4.weight | 파라미터 Shape: (1, 2) | Gradient Shape: (1, 2) | Gradient 크기: 0.00328
# 4.bias | 파라미터 Shape: (1,) | Gradient Shape: (1,) | Gradient 크기: 0.11058
# 역전파 후 가중치 변경 여부: False

입력 Shape: (4, 3)
Logit Shape: (4, 1)
손실: 0.7115
0.weight | 파라미터 Shape: (4, 3) | Gradient Shape: (4, 3) | Gradient 크기: 0.06920
0.bias | 파라미터 Shape: (4,) | Gradient Shape: (4,) | Gradient 크기: 0.05926
2.weight | 파라미터 Shape: (2, 4) | Gradient Shape: (2, 4) | Gradient 크기: 0.04049
2.bias | 파라미터 Shape: (2,) | Gradient Shape: (2,) | Gradient 크기: 0.04378
4.weight | 파라미터 Shape: (1, 2) | Gradient Shape: (1, 2) | Gradient 크기: 0.00328
4.bias | 파라미터 Shape: (1,) | Gradient Shape: (1,) | Gradient 크기: 0.11058
역전파 후 가중치 변경 여부: False


In [5]:
import torch
import torch.nn as nn

# ============================================================
# 전체 흐름 한눈에 보기
#   1. 데이터 준비 (X: 입력, y: 정답)
#   2. 모델 정의 (3 → 4 → 2 → 1 DNN)
#   3. 순전파 (X → Logit → Loss)
#   4. 역전파 (Loss → 각 파라미터의 Gradient 계산)
#   5. Gradient 확인 (.grad에 잘 저장됐는지)
#   6. 가중치가 바뀌었는지 확인 → 안 바뀜! (업데이트는 Optimizer 몫)
# ============================================================

# 난수 시드 고정
# nn.Linear는 생성될 때 가중치를 "랜덤"으로 초기화한다.
# 시드를 고정하면 매번 같은 초기 가중치가 나와서 결과가 재현된다.
torch.manual_seed(2)

# ----------------------
# 1. 입력 데이터와 실제 정답
# ----------------------

# X: 고객 4명 × 특성 3개  → Shape (4, 3)
#    행 = 고객 한 명 (데이터 1건), 열 = 특성 (방문, 장바구니, 체류 등)
#    4 = Batch Size, 3 = 입력 특성 수 (← 이 3이 nn.Linear(3, 4)의 3과 맞아야 함)
X = torch.tensor([
    [0.5, 0.3, 0.8],   # 고객 1
    [0.2, 0.0, 0.3],   # 고객 2
    [0.8, 0.4, 1.0],   # 고객 3
    [0.3, 0.1, 0.5]    # 고객 4
], dtype=torch.float32)  # 신경망 연산은 실수형(float32)이어야 함

# y: 고객별 실제 구매 여부 (1 = 구매, 0 = 미구매) → Shape (4, 1)
#    모델 출력 logits도 (4, 1)이라서 Shape를 맞춰둔 것
#    (BCEWithLogitsLoss는 입력과 정답의 Shape가 같아야 함)
y = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [0.0]
])

# ----------------------
# 2. DNN 모델 정의
# ----------------------

# nn.Sequential: 안에 넣은 층을 "순서대로" 통과시키는 컨테이너
# 각 층에는 0부터 번호(인덱스)가 붙는다 → 아래 named_parameters()의 "0.", "2.", "4."가 이 번호
model = nn.Sequential(
    nn.Linear(3, 4),   # [0] 은닉층 1: 특성 3개 → 뉴런 4개
                       #     weight (4, 3), bias (4,)  ← PyTorch는 (출력, 입력) 순서로 저장
                       #     파라미터 3×4 + 4 = 16개
    nn.ReLU(),         # [1] 음수 → 0 (비선형성 추가). 파라미터 없음

    nn.Linear(4, 2),   # [2] 은닉층 2: 4개 → 2개
                       #     weight (2, 4), bias (2,)  → 파라미터 10개
    nn.ReLU(),         # [3] 파라미터 없음

    nn.Linear(2, 1)    # [4] 출력층: 2개 → 1개 (구매 점수 = Logit)
                       #     weight (1, 2), bias (1,)  → 파라미터 3개
)                      # 전체 학습 파라미터: 16 + 10 + 3 = 29개
# 주의: 마지막에 Sigmoid가 없다!
#       아래 BCEWithLogitsLoss가 내부에서 Sigmoid까지 처리하기 때문
#       → 모델은 확률이 아니라 Logit(아무 범위의 점수)을 출력한다

# 이진 분류 손실 함수
# BCEWithLogitsLoss = Sigmoid + BCE를 한 번에, 수치적으로 안정적으로 계산
# 입력으로 "확률"이 아니라 "Logit"을 받는다
# 기본 설정(reduction='mean')이라 고객 4명의 손실을 "평균"낸 값 하나를 반환
loss_fn = nn.BCEWithLogitsLoss()

# 학습 모드로 설정
# 이 모델엔 Dropout·BatchNorm이 없어서 실제 차이는 없지만,
# 학습 코드에선 습관적으로 넣어주는 것이 좋다 (예측할 땐 model.eval())
model.train()

# ----------------------
# 3. 순전파
# ----------------------

# 기존 Gradient 초기화
# PyTorch는 backward() 할 때마다 Gradient를 .grad에 "누적(더하기)"한다.
# 이전에 남은 값이 있으면 섞이므로 매번 0으로 비우고 시작
# (Optimizer를 쓸 땐 보통 optimizer.zero_grad()로 대신함)
model.zero_grad()

# 역전파 "전" 가중치를 따로 복사해서 저장 (나중에 바뀌었는지 비교하려고)
#   model[0].weight : 첫 번째 Linear의 가중치, Shape (4, 3)
#   .detach()       : 계산 그래프(Gradient 추적)에서 떼어낸 텐서로 만듦
#   .clone()        : 메모리까지 완전히 복사
#                     → clone이 없으면 원본과 같은 메모리를 가리켜서
#                       나중에 원본이 바뀌면 weight_before도 같이 바뀌어 비교가 무의미해짐
weight_before = model[0].weight.detach().clone()

# 순전파: X가 층을 순서대로 통과
#   X (4, 3)
#   → Linear(3,4) → (4, 4)
#   → ReLU        → (4, 4)
#   → Linear(4,2) → (4, 2)
#   → ReLU        → (4, 2)
#   → Linear(2,1) → (4, 1)   ← 고객 4명 각각의 Logit
# 이 과정에서 PyTorch는 중간값(가중합, ReLU 출력 등)과 연산 관계를
# "계산 그래프"에 기록해 둔다 → 역전파 때 이 기록을 거꾸로 따라감
logits = model(X)

# 손실 계산: Logit과 정답을 비교
# 내부적으로: Sigmoid(logits) → 확률 p → 고객별 BCE → 평균
# 결과는 숫자 하나(스칼라 텐서) → 그래서 바로 backward() 가능
loss = loss_fn(logits, y)

print("입력 Shape:", tuple(X.shape))          # (4, 3)
print("Logit Shape:", tuple(logits.shape))    # (4, 1)
print("손실:", round(loss.item(), 4))           # .item(): 스칼라 텐서 → 파이썬 숫자

# ----------------------
# 4. 역전파
# ----------------------

# loss에서 출발해 계산 그래프를 거꾸로 따라가며 Chain Rule로
# "모든 학습 파라미터"에 대한 ∂Loss/∂파라미터를 계산
#   출력층(4) → ReLU(3) → 은닉층2(2) → ReLU(1) → 은닉층1(0) 순서
# 계산 결과는 각 파라미터의 .grad 속성에 저장된다
# ※ 여기서는 Gradient "계산만" 한다. 가중치 값은 건드리지 않음!
loss.backward()

# ----------------------
# 5. Gradient 확인
# ----------------------

# named_parameters(): (이름, 파라미터) 쌍을 하나씩 꺼내줌
#   이름 형식: "층 인덱스.파라미터 종류" → 예) "0.weight", "4.bias"
#   ReLU(1, 3번)는 파라미터가 없어서 목록에 안 나온다
for name, parameter in model.named_parameters():
    print(
        f"{name} | "
        f"파라미터 Shape: {tuple(parameter.shape)} | "
        # Gradient Shape == 파라미터 Shape
        #   → 가중치 원소 하나하나마다 "이걸 바꾸면 손실이 얼마나 변하나"가 계산되기 때문
        #   → 그래서 나중에 W - lr × grad 를 자리끼리 바로 계산할 수 있음
        f"Gradient Shape: {tuple(parameter.grad.shape)} | "
        # .norm(): Gradient 원소들의 L2 Norm (= √(각 원소²의 합))
        #   여러 숫자를 "전체 크기 하나"로 요약 → 층별 Gradient 크기 비교용
        #   (너무 작으면 기울기 소실, 너무 크면 폭주를 의심해볼 수 있음.
        #    단, 한 번 값만으로 판단하진 않음)
        f"Gradient 크기: {parameter.grad.norm().item():.5f}"
    )

# ----------------------
# 6. 파라미터 변경 여부 확인
# ----------------------

# 역전파 "후" 첫 번째 층 가중치
# (비교만 할 거라 clone 없이 detach만 해도 충분)
weight_after = model[0].weight.detach()

# torch.equal: 두 텐서의 Shape와 모든 값이 같으면 True
# 같으면 → 안 바뀐 것 → is_changed = False
is_changed = not torch.equal(
    weight_before,
    weight_after
)

# 결과: False
# 이유: backward()는 Gradient를 .grad에 "저장"만 할 뿐, 가중치를 바꾸지 않는다
#       가중치를 실제로 바꾸려면 Optimizer가 필요:
#         optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
#         optimizer.step()   # W ← W - lr × W.grad  → 이때 비로소 True가 됨
print("역전파 후 가중치 변경 여부:", is_changed)

# ============================================================
# 실행 결과 읽는 법
#
# 입력 Shape: (4, 3)
# Logit Shape: (4, 1)
# 손실: 0.7115
#   → 학습 전 랜덤 가중치라 거의 "찍는" 수준
#     (참고: 확률을 전부 0.5로 예측하면 BCE = -ln(0.5) ≈ 0.693)
#
# 0.weight | (4, 3) | (4, 3) | 0.06920   ← 은닉층1 가중치 12개의 Gradient 크기
# 0.bias   | (4,)   | (4,)   | 0.05926
# 2.weight | (2, 4) | (2, 4) | 0.04049   ← 은닉층2
# 2.bias   | (2,)   | (2,)   | 0.04378
# 4.weight | (1, 2) | (1, 2) | 0.00328   ← 출력층 가중치: 유독 작음
# 4.bias   | (1,)   | (1,)   | 0.11058   ← 출력층 편향: 가장 큼
#   → 출력층 가중치 Gradient = (p - y) × "그 층에 들어온 입력(은닉층2 ReLU 출력)"
#     은닉층2 출력이 0이거나 작으면 이 값도 작아진다 (ReLU가 꺼진 뉴런 영향 가능)
#   → 출력층 편향 Gradient = (p - y)의 평균이라 입력 크기 영향을 안 받음
#   → 이 숫자들은 "한 번" 계산한 값이라 이것만으로 소실/폭주를 판단하진 않음
#
# 역전파 후 가중치 변경 여부: False
#   → backward()는 계산만, 업데이트는 optimizer.step()
# ============================================================

# ------------------------------------------------------------
# 참고: 실제 학습 루프라면 이렇게 반복한다
#
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# for epoch in range(100):
#     optimizer.zero_grad()          # 1. Gradient 초기화
#     logits = model(X)              # 2. 순전파
#     loss = loss_fn(logits, y)      # 3. 손실 계산
#     loss.backward()                # 4. 역전파 (Gradient 계산)
#     optimizer.step()               # 5. 가중치 업데이트 ← 이게 빠져 있던 것
#
# 예측할 때:
# model.eval()
# with torch.no_grad():              # Gradient 추적 끔 (메모리 절약)
#     probs = torch.sigmoid(model(X))  # Logit → 확률 (손실 함수 밖이라 직접 적용)
#     preds = (probs >= 0.5).int()     # Threshold로 클래스 결정
# ------------------------------------------------------------

입력 Shape: (4, 3)
Logit Shape: (4, 1)
손실: 0.7115
0.weight | 파라미터 Shape: (4, 3) | Gradient Shape: (4, 3) | Gradient 크기: 0.06920
0.bias | 파라미터 Shape: (4,) | Gradient Shape: (4,) | Gradient 크기: 0.05926
2.weight | 파라미터 Shape: (2, 4) | Gradient Shape: (2, 4) | Gradient 크기: 0.04049
2.bias | 파라미터 Shape: (2,) | Gradient Shape: (2,) | Gradient 크기: 0.04378
4.weight | 파라미터 Shape: (1, 2) | Gradient Shape: (1, 2) | Gradient 크기: 0.00328
4.bias | 파라미터 Shape: (1,) | Gradient Shape: (1,) | Gradient 크기: 0.11058
역전파 후 가중치 변경 여부: False


In [6]:
import torch

# ----------------------
# 기울기 전달 확인 함수
# ----------------------

def inspect_gradient(scale, depth=8):
    # Gradient를 추적할 입력값
    x = torch.tensor(1.0, requires_grad=True)

    # 단계별 출력값 기록
    values = [x]

    h = x

    # 동일한 연산을 여러 번 수행
    for _ in range(depth):
        h = h * scale

        # 중간 출력의 Gradient도 확인
        h.retain_grad()
        values.append(h)

    # 목표값과 손실 계산
    target = torch.tensor(1.0)
    loss = (h - target) ** 2

    # 역전파
    loss.backward()

    # 결과 확인
    print(f"\n곱하는 값: {scale}")
    print(f"연산 단계: {depth}")
    print(f"최종 출력: {h.item():.6f}")
    print(f"손실: {loss.item():.6f}")

    print(
        "마지막 출력 Gradient:",
        f"{values[-1].grad.item():.6f}"
    )

    print(
        "첫 번째 단계 Gradient:",
        f"{values[1].grad.item():.6f}"
    )

    print(
        "입력 Gradient:",
        f"{x.grad.item():.6f}"
    )


# ----------------------
# 1. 작은 값이 반복되는 경우
# ----------------------

inspect_gradient(scale=0.5)

# ----------------------
# 2. 큰 값이 반복되는 경우
# ----------------------

inspect_gradient(scale=2.0)


# 곱하는 값: 0.5
# 연산 단계: 8
# 최종 출력: 0.003906
# 손실: 0.992203
# 마지막 출력 Gradient: -1.992188
# 첫 번째 단계 Gradient: -0.015564
# 입력 Gradient: -0.007782

# 곱하는 값: 2.0
# 연산 단계: 8
# 최종 출력: 256.000000
# 손실: 65025.000000
# 마지막 출력 Gradient: 510.000000
# 첫 번째 단계 Gradient: 65280.000000
# 입력 Gradient: 130560.000000


곱하는 값: 0.5
연산 단계: 8
최종 출력: 0.003906
손실: 0.992203
마지막 출력 Gradient: -1.992188
첫 번째 단계 Gradient: -0.015564
입력 Gradient: -0.007782

곱하는 값: 2.0
연산 단계: 8
최종 출력: 256.000000
손실: 65025.000000
마지막 출력 Gradient: 510.000000
첫 번째 단계 Gradient: 65280.000000
입력 Gradient: 130560.000000


In [7]:
import torch
import torch.nn as nn

# 재현 가능한 결과
torch.manual_seed(42)

# ----------------------
# 1. 입력 데이터
# ----------------------

X = torch.tensor([
    [0.5, 0.3, 0.8],
    [0.2, 0.0, 0.3],
    [0.8, 0.4, 1.0],
    [0.3, 0.1, 0.5]
], dtype=torch.float32)

y = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [0.0]
])

# ----------------------
# 2. DNN 모델 정의
# ----------------------

model = nn.Sequential(
    nn.Linear(3, 6),
    nn.ReLU(),

    nn.Linear(6, 4),
    nn.ReLU(),

    nn.Linear(4, 1)
)

# 손실 함수
loss_fn = nn.BCEWithLogitsLoss()

# ----------------------
# 3. 순전파와 손실 계산
# ----------------------

model.zero_grad()

# 가중치 변경 여부 확인용
weight_before = model[0].weight.detach().clone()

logits = model(X)
loss = loss_fn(logits, y)

print("손실:", round(loss.item(), 4))

# ----------------------
# 4. 역전파
# ----------------------

loss.backward()

# ----------------------
# 5. 레이어별 Gradient 확인
# ----------------------

for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        grad_norm = parameter.grad.norm().item()

        print(
            f"{name} | "
            f"Gradient Norm: {grad_norm:.5f}"
        )

# ----------------------
# 6. Gradient Clipping
# ----------------------

# Clipping 전 전체 Gradient Norm
grads = [
    p.grad
    for p in model.parameters()
    if p.grad is not None
]

total_norm_before = torch.linalg.vector_norm(
    torch.stack([grad.norm() for grad in grads])
)

# 설명을 위한 작은 기준값
max_norm = 0.03

# Gradient Clipping 적용
original_norm = nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=max_norm
)

# Clipping 후 전체 Gradient Norm
total_norm_after = torch.linalg.vector_norm(
    torch.stack([grad.norm() for grad in grads])
)

print(
    "Clipping 전:",
    round(total_norm_before.item(), 5)
)

print(
    "Clipping 후:",
    round(total_norm_after.item(), 5)
)

# ----------------------
# 7. 가중치 변경 여부 확인
# ----------------------

weight_changed = not torch.equal(
    weight_before,
    model[0].weight.detach()
)

print("가중치 변경 여부:", weight_changed)

# 실행 결과 예시
# 손실: 0.6871
# 0.weight | Gradient Norm: 0.02170
# 0.bias | Gradient Norm: 0.01429
# 2.weight | Gradient Norm: 0.02602
# 2.bias | Gradient Norm: 0.01865
# 4.weight | Gradient Norm: 0.04375
# 4.bias | Gradient Norm: 0.04481
# Clipping 전: 0.07498
# Clipping 후: 0.03
# 가중치 변경 여부: False

손실: 0.6871
0.weight | Gradient Norm: 0.02170
0.bias | Gradient Norm: 0.01429
2.weight | Gradient Norm: 0.02602
2.bias | Gradient Norm: 0.01865
4.weight | Gradient Norm: 0.04375
4.bias | Gradient Norm: 0.04481
Clipping 전: 0.07498
Clipping 후: 0.03
가중치 변경 여부: False


In [1]:
board = [
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
]

for row in range(len(board)):
    for col in range(len(board[0])):
        if board[row][col] == 0:
            print(row, col)

0 0
0 2
1 1
2 0


In [4]:
def solution(seats):
  rows = len(seats)
  cols = len(seats[0])

  directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 상, 하, 좌, 우
  answer = [-1, -1]
  min_noise = 5

  for row in range(rows):
    for col in range(cols):
      if seats[row][col] == 0:
        noise = 0

        for dr, dc in directions:
          next_row = dr + row
          next_col = dc + col

          if 0 <= next_row < rows and 0 <= next_col < cols:
            if seats[next_row][next_col] == 1:
              noise += 1

        if noise <= min_noise:
          min_noise = noise
          answer = [row, col]

  return answer


print(solution([[1, 0, 1], [0, 0, 0], [1, 0, 1]]))

[1, 1]


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# 1. 이미지 데이터 준비
# 클래스 0: 가로선
# 클래스 1: 세로선

X = torch.tensor([
    # 클래스 0: 가로선
    [
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]
    ],
    # 클래스 0: 가로선
    [
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]
    ],
    # 클래스 1: 세로선
    [
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0]
    ],
    # 클래스 1: 세로선
    [
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0]
    ]
], dtype=torch.float32).unsqueeze(1)

y = torch.tensor([
    [0], [0], [1], [1]
], dtype=torch.float32)

print("입력 Shape:", tuple(X.shape))
print("정답 Shape:", tuple(y.shape))


# 2. CNN 모델 정의
model = nn.Sequential(
    nn.Conv2d(   # Conv layer 정의 
        in_channels=1,
        out_channels=2,
        kernel_size=3
    ),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2), # Pooling 정의 
    nn.Flatten(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

model.train()

# 5 손실 함수와 Optimizer
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)


# CNN 학습
for epoch in range(501):

    # 이전 기울기 초기화
    optimizer.zero_grad()

    # 순전파: Conv → ReLU → Pool → FC
    outputs = model(X)

    # 6.1 손실 계산
    loss = criterion(outputs, y)

    # 6.2 역전파
    loss.backward()

    # 가중치 업데이트
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, "
            f"Loss: {loss.item():.4f}"
        )


# 7. 학습 완료 후 예측
model.eval()

with torch.no_grad():
    probabilities = model(X)
    predictions = (probabilities >= 0.5).int()

print("예측:", predictions.squeeze().tolist())
print("정답:", y.int().squeeze().tolist())


# 각 레이어의 출력 Shape 확인
with torch.no_grad():
    h = X

    print("입력:", tuple(h.shape))

    for layer in model:
        h = layer(h)
        print(
            layer.__class__.__name__,
            tuple(h.shape)
        )

입력 Shape: (4, 1, 6, 6)
정답 Shape: (4, 1)
Epoch 0, Loss: 0.7067
Epoch 100, Loss: 0.0000
Epoch 200, Loss: 0.0000
Epoch 300, Loss: 0.0000
Epoch 400, Loss: 0.0000
Epoch 500, Loss: 0.0000
예측: [0, 0, 1, 1]
정답: [0, 0, 1, 1]
입력: (4, 1, 6, 6)
Conv2d (4, 2, 4, 4)
ReLU (4, 2, 4, 4)
MaxPool2d (4, 2, 2, 2)
Flatten (4, 8)
Linear (4, 1)
Sigmoid (4, 1)


In [6]:
import numpy as np

# ----------------------
# 1. 입력 이미지와 Filter
# ----------------------

image = np.array([
    [1, 0, 1, 0],
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 1]
])

kernel = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
])

# ----------------------
# 2. Convolution 함수
# ----------------------

def conv2d(image, kernel, stride=1, padding=0):
    # Padding 적용
    padded_image = np.pad(
        image,
        ((padding, padding), (padding, padding)),
        mode="constant"
    )

    kernel_h, kernel_w = kernel.shape
    input_h, input_w = padded_image.shape

    # 출력 크기 계산
    output_h = (input_h - kernel_h) // stride + 1
    output_w = (input_w - kernel_w) // stride + 1

    output = np.zeros((output_h, output_w))

    # Filter를 이동시키며 계산
    for i in range(output_h):
        for j in range(output_w):
            start_h = i * stride
            start_w = j * stride

            # Filter가 적용될 영역
            region = padded_image[
                start_h:start_h + kernel_h,
                start_w:start_w + kernel_w
            ]

            # 원소별 곱셈 후 합산
            output[i, j] = np.sum(region * kernel)

    return output

# ----------------------
# 3. 기본 Convolution
# ----------------------

feature_map = conv2d(
    image,
    kernel,
    stride=1,
    padding=0
)

print("입력 Shape:", image.shape)
print("Feature Map:")
print(feature_map)
print("출력 Shape:", feature_map.shape)

# ----------------------
# 4. Stride 변경
# ----------------------

stride_output = conv2d(
    image,
    kernel,
    stride=2,
    padding=0
)

print("Stride=2 출력 Shape:", stride_output.shape)

# ----------------------
# 5. Padding 변경
# ----------------------

padding_output = conv2d(
    image,
    kernel,
    stride=1,
    padding=1
)

print("Padding=1 출력 Shape:", padding_output.shape)

# 실행 결과
# 입력 Shape: (4, 4)
# Feature Map:
# [[3. 0.]
#  [3. 3.]]
# 출력 Shape: (2, 2)
# Stride=2 출력 Shape: (1, 1)
# Padding=1 출력 Shape: (4, 4)

입력 Shape: (4, 4)
Feature Map:
[[3. 0.]
 [3. 3.]]
출력 Shape: (2, 2)
Stride=2 출력 Shape: (1, 1)
Padding=1 출력 Shape: (4, 4)


In [7]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ----------------------
# 1. 입력 이미지
# ----------------------

# 이미지 4개, 흑백 채널 1개, 28x28
# 구조 확인을 위한 임의 데이터
X = torch.randn(4, 1, 28, 28)

# ----------------------
# 2. CNN 모델 정의
# ----------------------

model = nn.Sequential(
    # 첫 번째 특징 추출
    nn.Conv2d(
        in_channels=1,
        out_channels=4,
        kernel_size=3,
        stride=1,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    # 두 번째 특징 추출
    nn.Conv2d(
        in_channels=4,
        out_channels=8,
        kernel_size=3,
        stride=1,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    # 분류층
    nn.Flatten(),
    nn.Linear(8 * 7 * 7, 10)
)

# ----------------------
# 3. 순전파
# ----------------------

model.eval()

with torch.no_grad():
    # 첫 번째 Convolution 블록
    h1 = model[:3](X)

    # 두 번째 Convolution 블록
    h2 = model[3:6](h1)

    # Flatten
    flat = model[6](h2)

    # 최종 Logit
    logits = model[7](flat)

    # 클래스별 확률
    probabilities = torch.softmax(logits, dim=1)

    # 최종 예측 클래스
    predictions = torch.argmax(logits, dim=1)

# ----------------------
# 4. 결과 확인
# ----------------------

print("입력 Shape:", tuple(X.shape))
print("Conv Block 1:", tuple(h1.shape))
print("Conv Block 2:", tuple(h2.shape))
print("Flatten:", tuple(flat.shape))
print("Logit:", tuple(logits.shape))
print("확률:", tuple(probabilities.shape))
print("예측 클래스 Shape:", tuple(predictions.shape))

print(
    "첫 번째 이미지의 확률 합:",
    round(probabilities[0].sum().item(), 4)
)

# 입력 Shape: (4, 1, 28, 28)
# Conv Block 1: (4, 4, 14, 14)
# Conv Block 2: (4, 8, 7, 7)
# Flatten: (4, 392)
# Logit: (4, 10)
# 확률: (4, 10)
# 예측 클래스 Shape: (4,)
# 첫 번째 이미지의 확률 합: 1.0

입력 Shape: (4, 1, 28, 28)
Conv Block 1: (4, 4, 14, 14)
Conv Block 2: (4, 8, 7, 7)
Flatten: (4, 392)
Logit: (4, 10)
확률: (4, 10)
예측 클래스 Shape: (4,)
첫 번째 이미지의 확률 합: 1.0


In [8]:
numbers = [10, 20, 30, 40, 50, 60, 70]
target = 60

left = 0
right = len(numbers) - 1

answer = -1

while left <= right:
    mid = (left + right) // 2

    if numbers[mid] == target:
        answer = mid
        break

    elif numbers[mid] < target:
        left = mid + 1

    else:
        right = mid - 1

print(answer)

5


In [ ]:
# 이분탐색 코테 코드
def solution(data_ids, queries):
    result = []
    for target in queries:
        left = 0
        right = len(data_ids) - 1
        answer = len(data_ids) + 1 # 인덱스 번호
        while left <= right:
            mid = (left + right) // 2

            if data_ids[mid] == target:
                answer = mid
                break
            elif data_ids[mid] < target:
                left = mid + 1
            else:
                right = mid - 1

        result.append(answer)

    return result


print(solution([10, 20, 30, 40, 50], [40, 10, 20]))


[3, 0, 1]


In [15]:
import numpy as np

# 1. 세 개 정보의 Value 벡터
values = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

# 2. Attention Weight
weights = np.array([0.1, 0.7, 0.2])

# 3. 각 벡터에 가중치 적용 후 합산
weighted_values = values * weights[:, None]

attention_output = weighted_values.sum(axis=0)

# 4. 결과 확인
print("Attention Weight:", weights)
print("가중치 합:", weights.sum())
print("Attention Output:", attention_output)

# 실행 결과
# Attention Weight: [0.1 0.7 0.2]
# 가중치 합: 1.0
# Attention Output: [0.3 0.9]

Attention Weight: [0.1 0.7 0.2]
가중치 합: 1.0
Attention Output: [0.3 0.9]


In [16]:
import numpy as np

# ----------------------
# 1. Query, Key, Value
# ----------------------

# 현재 처리할 위치의 Query
Q = np.array([1.0, 0.0])

# 세 위치의 Key
K = np.array([
    [0.0, 1.0],  # 철수는
    [1.0, 0.0],  # 책을
    [0.0, 1.0]   # 그것을
])

# 세 위치의 Value
V = np.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [1.0, 1.0]
])

# ----------------------
# 2. 관련성 점수 계산
# ----------------------

scores = K @ Q

# ----------------------
# 3. Attention Weight
# ----------------------

# Softmax
exp_scores = np.exp(scores - np.max(scores))
weights = exp_scores / exp_scores.sum()

# ----------------------
# 4. Value 가중합
# ----------------------

output = weights @ V

# ----------------------
# 5. 결과 확인
# ----------------------

print("Attention Score:", scores)
print("Attention Weight:", np.round(weights, 3))
print("Attention Output:", np.round(output, 3))

# 실행 결과
# Attention Score: [0. 1. 0.]
# Attention Weight: [0.212 0.576 0.212]
# Attention Output: [0.424 1.364]

Attention Score: [0. 1. 0.]
Attention Weight: [0.212 0.576 0.212]
Attention Output: [0.424 1.364]


In [17]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ① 시계열 데이터 준비
# 상승: 1, 하락: 0
X = torch.tensor([
    [0.1, 0.3, 0.5, 0.7],
    [0.2, 0.4, 0.6, 0.8],
    [0.0, 0.2, 0.4, 0.6],
    [0.3, 0.4, 0.5, 0.6],
    [0.7, 0.5, 0.3, 0.1],
    [0.8, 0.6, 0.4, 0.2],
    [0.6, 0.4, 0.2, 0.0],
    [0.6, 0.5, 0.4, 0.3],
], dtype=torch.float32).unsqueeze(-1)

y = torch.tensor([
    [1], [1], [1], [1],
    [0], [0], [0], [0]
], dtype=torch.float32)


# ② Transformer 분류기 정의
class TransformerClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_projection = nn.Linear(1, 16)
        self.pos_embedding = nn.Embedding(4, 16)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=16,
            nhead=4,
            dim_feedforward=32,
            dropout=0.0,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1
        )

        self.classifier = nn.Linear(16, 1)

    def forward(self, x):
        # ③ 입력 투영 + 위치 정보 추가
        positions = torch.arange(
            x.size(1), device=x.device
        )

        h = self.input_projection(x)
        h = h + self.pos_embedding(positions).unsqueeze(0)

        # ④ Transformer 순전파
        h = self.encoder(h)

        # 모든 시점의 출력 평균
        pooled = h.mean(dim=1)

        # 최종 분류 점수 (Logit)
        logits = self.classifier(pooled)

        return logits


model = TransformerClassifier()

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.01
)


# ⑤~⑥ 모델 학습
model.train()

for epoch in range(1, 301):

    optimizer.zero_grad()

    logits = model(X)

    # 손실 계산
    loss = criterion(logits, y)

    # 역전파
    loss.backward()

    # 가중치 업데이트
    optimizer.step()

    if epoch in (1, 100, 200, 300):
        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {loss.item():.6f}"
        )


# ⑦ 예측
model.eval()

with torch.no_grad():
    logits = model(X)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).int()

print("예측:", predictions.squeeze(1).tolist())
print("정답:", y.int().squeeze(1).tolist())

Epoch   1 | Loss: 0.739156
Epoch 100 | Loss: 0.000635
Epoch 200 | Loss: 0.000283
Epoch 300 | Loss: 0.000157
예측: [1, 1, 1, 1, 0, 0, 0, 0]
정답: [1, 1, 1, 1, 0, 0, 0, 0]


In [ ]:
import torch
import torch.nn as nn

# 데이터 준비
sentences = [
    '이 영화 정말 좋네',
    '배우 퀄리티가 진짜 좋다 ㅋㅋ',
    '잘잤다',
    '왜케 지루하냐~'
]
labels = torch.tensor([1, 1, 0, 0])
PAD = 0
vocab = {'<PAD>': PAD}
for s in sentences:
    for tok in s.split():
        vocab.setdefault(tok, len(vocab))

def encode(s, max_len = 6):
    ids = [vocab[t] for t in s.split()]
    return ids + [PAD] * (max_len - len(ids))
X= torch.tensor([encode(s) for s in sentences])

class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model = 32, nhead = 4, num_layer = 2,
                 num_classes = 2, max_len = 6):
        super().__init__()

        self.tok_emb = nn.Embedding(vocab_size, d_model,padding_idx=PAD)
        self.pod_emb = nn.Embedding(max_len, d_model)
        

In [ ]:



# =========================================================
# 1. 모델 정의
# =========================================================
class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2,
                 num_classes=2, max_len=6):
        super().__init__()

        # ② Token Embedding: 번호 → 의미 벡터 ("무엇")
        #    (vocab_size × d_model) 표에서 행을 꺼냄, 값은 학습됨
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)

        # ③ Positional Embedding: 위치 번호 → 위치 벡터 ("어디")
        #    여기선 학습형(Learned) 방식. 원 논문은 Sin·Cos 고정값
        self.pos_emb = nn.Embedding(max_len, d_model)

        # ④~⑥ Encoder 블록 하나 = Multi-Head Self-Attention + FFN
        #      (Residual + LayerNorm, Dropout까지 내부에 다 들어 있음)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,          # 토큰 벡터 크기 (끝까지 유지됨)
            nhead=nhead,              # Head 수 → Head당 32/4 = 8차원
            dim_feedforward=4 * d_model,  # FFN 중간 크기 (보통 4배)
            dropout=0.1,
            batch_first=True,         # 입력을 (Batch, 토큰 수, d_model) 순서로
        )
        # ⑦ 블록을 num_layers번 쌓기 (그림의 N×)
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=num_layers, enable_nested_tensor=False
        )

        # 분류층: 문장 벡터 → 클래스별 점수(Logit)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, ids):                       # ids: (B, T)
        B, T = ids.shape

        # Padding Mask: <PAD> 위치는 Attention에서 무시 (True = 무시)
        pad_mask = (ids == PAD)                   # (B, T)

        # ②+③ 임베딩 + 위치 정보 더하기
        positions = torch.arange(T, device=ids.device)        # [0, 1, ..., T-1]
        h = self.tok_emb(ids) + self.pos_emb(positions)       # (B, T, d_model)

        # ④~⑦ Encoder 통과 → 토큰마다 문맥이 담긴 벡터
        h = self.encoder(h, src_key_padding_mask=pad_mask)    # (B, T, d_model)

        # 문장 하나로 요약: PAD를 뺀 토큰들의 평균 (Many-to-One)
        keep = (~pad_mask).unsqueeze(-1).float()              # (B, T, 1) 진짜 토큰=1
        sent = (h * keep).sum(dim=1) / keep.sum(dim=1)        # (B, d_model)

        return self.classifier(sent)                          # (B, num_classes) Logit


# =========================================================
# 2. 학습
# =========================================================
torch.manual_seed(0)
model = EncoderClassifier(vocab_size=len(vocab))
loss_fn = nn.CrossEntropyLoss()                 # Logit을 그대로 받음 (Softmax 내장)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

for epoch in range(100):
    model.train()                    # Dropout 켜기
    optimizer.zero_grad()            # Gradient 초기화
    logits = model(X)                # 순전파 (4, 2)
    loss = loss_fn(logits, labels)   # 손실
    loss.backward()                  # 역전파 (Attention의 W_Q·W_K·W_V까지 전부)
    optimizer.step()                 # 업데이트
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

# =========================================================
# 3. 예측
# =========================================================
model.eval()                         # Dropout 끄기
with torch.no_grad():
    test = torch.tensor([encode("정말 좋았다")])
    prob = torch.softmax(model(test), dim=1)   # 확률이 필요할 때만 Softmax
    print("부정/긍정 확률:", prob)

epoch   0 | loss 0.7827
epoch  20 | loss 0.0572
epoch  40 | loss 0.0104
epoch  60 | loss 0.0065
epoch  80 | loss 0.0051
부정/긍정 확률: tensor([[0.9881, 0.0119]])


In [4]:
available = [6, 2, 4]
need = 3

available.sort()

for size in available:
    if size >= need:
        print(size)
        break

4


In [ ]:
def packages(packages, lockers):
    packages.sort()
    lockers.sort()

    i, j, count = 0, 0, 0

    while i <len(packages) and j < len(lockers):
        if packages[i] <= lockers[j]:
            count += 1
            i += 1
            j += 1
        else: 
            j += 1

    return count

print(packages([4, 2, 6, 3], [3, 5, 4]))

3
